# AI-Driven Data Leakage Detection - Exploratory Analysis

This notebook provides exploratory data analysis and model experimentation.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 1. Load Dataset

In [ ]:
from config import DATASET_PATH

# Load data
df = pd.read_csv(DATASET_PATH)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 2. Data Overview

In [ ]:
# Basic info
print("Dataset Info:")
df.info()

print("\nBasic Statistics:")
df.describe()

In [ ]:
# Class distribution
print("Class Distribution:")
print(df['is_leakage'].value_counts())
print("\nPercentage:")
print(df['is_leakage'].value_counts(normalize=True) * 100)

# Visualize
plt.figure(figsize=(8, 6))
df['is_leakage'].value_counts().plot(kind='bar', color=['green', 'red'])
plt.title('Class Distribution: Normal vs Leakage')
plt.xlabel('Class (0=Normal, 1=Leakage)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## 3. Feature Analysis

In [ ]:
# Numerical features comparison
numerical_cols = ['access_frequency', 'data_volume_mb', 'session_duration_mins', 
                  'hour_of_day', 'day_of_week']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    df.boxplot(column=col, by='is_leakage', ax=axes[idx])
    axes[idx].set_title(f'{col} by Class')
    axes[idx].set_xlabel('Class (0=Normal, 1=Leakage)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numerical_features = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numerical_features].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Categorical Features Analysis

In [ ]:
# Analyze categorical features
categorical_cols = ['user_role', 'department', 'data_sensitivity_level', 
                    'access_type', 'device_type']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, col in enumerate(categorical_cols):
    pd.crosstab(df[col], df['is_leakage'], normalize='index').plot(
        kind='bar', ax=axes[idx], stacked=False
    )
    axes[idx].set_title(f'{col} vs Leakage')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Proportion')
    axes[idx].legend(['Normal', 'Leakage'])

plt.tight_layout()
plt.show()

## 5. Temporal Patterns

In [ ]:
# Hour of day analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Normal activities
df[df['is_leakage'] == 0]['hour_of_day'].hist(bins=24, ax=ax1, color='green', alpha=0.7)
ax1.set_title('Normal Activities by Hour')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Count')

# Leakage activities
df[df['is_leakage'] == 1]['hour_of_day'].hist(bins=24, ax=ax2, color='red', alpha=0.7)
ax2.set_title('Leakage Activities by Hour')
ax2.set_xlabel('Hour of Day')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 6. Feature Engineering Preview

In [ ]:
from src.feature_engineering import FeatureEngineer

# Apply feature engineering
engineer = FeatureEngineer()
df_engineered = engineer.engineer_features(df.copy())

print("Engineered Features:")
print(engineer.engineered_features)

# Display sample
df_engineered[engineer.engineered_features].head()

## 7. Quick Model Test

In [ ]:
from src.preprocessing import DataPreprocessor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Preprocess
preprocessor = DataPreprocessor()
df_processed = preprocessor.preprocess(df_engineered, fit=True)
X_train, X_test, y_train, y_test = preprocessor.prepare_train_test_split(df_processed)

# Quick Random Forest test
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Leakage']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 8. Feature Importance

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='importance', y='feature', data=feature_importance.head(20))
plt.title('Top 20 Most Important Features')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

feature_importance.head(20)

## 9. Conclusion

This exploratory analysis reveals:
- Clear patterns distinguishing normal and leakage activities
- Temporal features show off-hours access for leakage
- Behavioral features (volume, frequency) are strong indicators
- Feature engineering significantly improves model performance
- Random Forest achieves strong baseline performance

Next steps:
1. Run full training pipeline: `python src/train_pipeline.py`
2. Launch web application: `python app/app.py`
3. Test predictions with different scenarios